In [ ]:
# =========================================
# Final pipeline: update "Listed Country"
# =========================================
IN_CSV     = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\hesta_pre_cleaned.csv"
OUT_CSV    = r"D:\LinhDao\Programming\SUPERFUNdProject\HestaSuper_Cleaned_ListedCountry-update.csv"
LOOKUP_CSV = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\yahoo_suffix_mapping_full.csv"

# Column names in your file
ISIN_COL           = "Stock ID"
NAME_COL           = "Name/Kind of Investment Item"   # change to "Name" if that's your header
LISTED_COUNTRY_COL = "Listed Country"

# yfinance concurrency
MAX_WORKERS = 16

# --- Optional check files (OFF by default) ---
GENERATE_CHECK_FILES = True  # ← set True only when you want the check CSVs
CHECK_OUT_SLIM  = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\HestaCHECK_yf_MIN.csv"
CHECK_NO_TICKER = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\HestaCHECK_yf_no_ticker.csv"

# -------------- imports ---------------
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    import yfinance as yf
except ImportError:
    raise SystemExit("Please install yfinance:  pip install yfinance")

# ---------- helpers ----------
def get_ticker_from_isin(isin: str) -> tuple[str, str]:
    """
    Returns (isin, resolved_ticker_or_empty).
    Touch .fast_info once to nudge yfinance to hit Yahoo, then read .ticker.
    """
    try:
        t = yf.Ticker(isin)
        try:
            _ = t.fast_info  # may raise; that's fine
        except Exception:
            pass
        sym = (t.ticker or "").strip()
        return isin, sym
    except Exception:
        return isin, ""

def read_lookup_frame(path: str) -> pd.DataFrame:
    for enc in ("utf-8-sig", "cp1252"):
        try:
            return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[], encoding=enc)
        except Exception:
            pass
    raise RuntimeError(f"Could not read lookup CSV: {path}")

def build_suffix_map(lookup_df: pd.DataFrame) -> tuple[dict, str]:
    """
    Returns:
      - suffix_map: dict 'SUFFIX' -> 'Country' (suffix normalized, no leading dot)
      - default_lc: Country from the FIRST ROW of the lookup file (for tickers with no dot)
    """
    if lookup_df.empty:
        return {}, "US"

    cols = {c.lower().strip(): c for c in lookup_df.columns}
    suffix_col  = cols.get("suffix")   or list(cols.values())[0]
    country_col = cols.get("country")  or list(cols.values())[2]

    default_lc = str(lookup_df.iloc[0][country_col]).strip()

    lk = lookup_df.copy()
    lk["_SUF"] = (
        lk[suffix_col].astype(str)
                      .str.strip()
                      .str.upper()
                      .str.lstrip(".")   # ".TO" -> "TO"
    )

    suffix_map = (
        lk[["_SUF", country_col]]
          .drop_duplicates(subset=["_SUF"])
          .set_index("_SUF")[country_col]
          .to_dict()
    )
    # ensure blank suffix maps to first-row country if present
    if lk.iloc[0]["_SUF"] == "":
        suffix_map[""] = default_lc

    return suffix_map, default_lc

def extract_suffix(symbol: str) -> str:
    if not isinstance(symbol, str): return ""
    s = symbol.strip()
    if "." not in s: return ""
    return s.rsplit(".", 1)[-1].upper()

def lc_for_symbol(symbol: str, suffix_map: dict, default_lc: str) -> str:
    if not isinstance(symbol, str) or not symbol.strip():
        return ""
    suf = extract_suffix(symbol)
    if suf == "":
        # no dot → treat as the first row in your suffix table
        return default_lc
    return suffix_map.get(suf, "")  # blank if unknown suffix

# ---------- OPTIONAL: check file writer (no summary here) ----------
def write_check_files(
    df_work: pd.DataFrame,
    *,
    name_col: str,
    isin_col: str,
    yf_col: str = "Yahoo ticker",
    lc_col: str = "LC",
    out_slim_path: str = CHECK_OUT_SLIM,
    out_no_ticker_path: str = CHECK_NO_TICKER,
):
    """
    Writes:
      - a slim check CSV (Name, ISIN, Yahoo ticker, LC)
      - a 'no ticker' CSV
    Assumes df_work has columns: [name_col, isin_col, yf_col, lc_col].
    """
    mask = df_work.get(isin_col, pd.Series([""] * len(df_work))).astype(str).str.strip().ne("")
    slim = df_work.loc[mask, [name_col, isin_col, yf_col, lc_col]].copy()
    slim = slim.rename(columns={name_col: "Name"})
    slim.to_csv(out_slim_path, index=False, encoding="utf-8-sig")
    print("Saved check (slim):", out_slim_path)

    no_ticker = slim[slim[yf_col].eq("")]
    if len(no_ticker):
        no_ticker.to_csv(out_no_ticker_path, index=False, encoding="utf-8-sig")
        print("Saved check (no ticker):", out_no_ticker_path, "| count:", len(no_ticker))

# -------------- main --------------
def main():
    # 1) Load full dataset
    df = pd.read_csv(IN_CSV, dtype=str, keep_default_na=False, na_values=[], encoding="cp1252")

    # Ensure Listed Country column exists, then CLEAR IT entirely (in-memory only)
    if LISTED_COUNTRY_COL not in df.columns:
        df[LISTED_COUNTRY_COL] = ""
    else:
        df[LISTED_COUNTRY_COL] = ""

    # Rows with non-empty ISIN
    mask = df.get(ISIN_COL, pd.Series([""]*len(df))).astype(str).str.strip().ne("")
    isins_series = df.loc[mask, ISIN_COL].astype(str).str.strip()

    # 2) Resolve Yahoo ticker from ISIN (de-dup + parallel)
    unique_isins = list(dict.fromkeys(isins_series))
    mapping = {}
    if unique_isins:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = [ex.submit(get_ticker_from_isin, i) for i in unique_isins]
            for n, fut in enumerate(as_completed(futs), 1):
                k, v = fut.result()
                mapping[k] = v
                if n % 200 == 0 or n == len(unique_isins):
                    print(f"{n}/{len(unique_isins)} ISINs processed")

    # 3) Build working frame (only rows we processed) to derive LC
    df_work = df.loc[mask, [NAME_COL, ISIN_COL]].copy()
    df_work["Yahoo ticker"] = isins_series.map(lambda s: mapping.get(s, ""))

    # 4) Build suffix map + default LC (first row’s country)
    lk = read_lookup_frame(LOOKUP_CSV)
    suffix_map, default_lc = build_suffix_map(lk)

    # 5) Derive LC per row (based on Yahoo ticker)
    df_work["LC"] = df_work["Yahoo ticker"].apply(lambda sym: lc_for_symbol(sym, suffix_map, default_lc))

    # 6) Write LC into the FULL dataset's "Listed Country" (no new columns in final file)
    lc_series = df_work["LC"]
    write_mask = lc_series.astype(str).str.strip().ne("")
    df.loc[df_work.index[write_mask], LISTED_COUNTRY_COL] = lc_series[write_mask].values

    # (Optional) small visibility on unmapped suffixes
    unmapped = (
        df_work.loc[df_work["Yahoo ticker"].astype(str).str.contains(r"\.", regex=True) & (df_work["LC"] == "")]
                ["Yahoo ticker"].map(extract_suffix).value_counts()
    )
    if not unmapped.empty:
        print("Unmapped Yahoo suffixes (top 20):")
        print(unmapped.head(20).to_string())

    # 7) Save final file
    df.to_csv(OUT_CSV, index=False, encoding="cp1252")
    print("✅ Final saved with Listed Country updated:", OUT_CSV)

    # 8) SUMMARY (in main, always printed)
    tickers = df_work["Yahoo ticker"].astype(str).str.strip()
    isins   = df_work[ISIN_COL].astype(str).str.strip()
    found_any = (tickers != "").sum()
    resolved  = ((tickers != "") & (tickers.str.upper() != isins.str.upper())).sum()
    echoed    = ((tickers != "") & (tickers.str.upper() == isins.str.upper())).sum()
    eligible  = len(df_work)
    not_found = eligible - found_any

    print("---- Summary ----")
    print(f"Rows with ISIN (processed): {eligible}")
    print(f"Returned any text:          {found_any}")
    print(f"Resolved to a real ticker:  {resolved}  (ticker != ISIN)")
    print(f"Echoed ISIN back:           {echoed}    (ticker == ISIN)")
    print(f"No ticker:                  {not_found}")
    if eligible:
        print(f"Hit rate (any):             {found_any/eligible*100:.2f}%")
        print(f"Hit rate (resolved):        {resolved/eligible*100:.2f}%")

    # 9) OPTIONAL: write check files (no summary here)
    if GENERATE_CHECK_FILES:
        write_check_files(
            df_work,
            name_col=NAME_COL,
            isin_col=ISIN_COL,
            yf_col="Yahoo ticker",
            lc_col="LC",
            out_slim_path=CHECK_OUT_SLIM,
            out_no_ticker_path=CHECK_NO_TICKER,
        )

if __name__ == "__main__":
    main()


200/2847 ISINs processed
400/2847 ISINs processed
600/2847 ISINs processed
800/2847 ISINs processed
1000/2847 ISINs processed
1200/2847 ISINs processed
1400/2847 ISINs processed
1600/2847 ISINs processed
1800/2847 ISINs processed
2000/2847 ISINs processed
2200/2847 ISINs processed
2400/2847 ISINs processed
2600/2847 ISINs processed
2800/2847 ISINs processed
2847/2847 ISINs processed
Unmapped Yahoo suffixes (top 20):
Yahoo ticker
CA    1
CL    1
✅ Final saved with Listed Country updated: D:\LinhDao\Programming\SUPERFUNdProject\HestaSuper_Cleaned_ListedCountry-update.csv
---- Summary ----
Rows with ISIN (processed): 2847
Returned any text:          1646
Resolved to a real ticker:  1646  (ticker != ISIN)
Echoed ISIN back:           0    (ticker == ISIN)
No ticker:                  1201
Hit rate (any):             57.82%
Hit rate (resolved):        57.82%
